In [1]:
# =========================
# ONE-CELL GROQ + MODEL TEST
# =========================

import os
import re
import ast
import joblib
import pandas as pd

try:
    from groq import Groq
except ImportError:
    raise ImportError("The Groq SDK is not installed. Run this once in a notebook cell: %pip install groq")

# -------------------------
# 1) CONFIG
# -------------------------
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or "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"
MODEL_PATH = "resume_job_matcher_model.pkl"
TRAINING_CSV_PATH = "training_pairs_debug.csv"
GROQ_MODEL = "llama-3.3-70b-versatile"

if GROQ_API_KEY == "PASTE_YOUR_GROQ_KEY_HERE":
    raise ValueError("Please paste your actual Groq API key into GROQ_API_KEY before running this cell.")

client = Groq(api_key=GROQ_API_KEY)

# -------------------------
# 2) LOAD SAVED FILES
# -------------------------
training_df = pd.read_csv(TRAINING_CSV_PATH)
model = joblib.load(MODEL_PATH)

# Convert list-like columns back from CSV strings to Python lists
def safe_parse_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    x = str(x).strip()
    if not x:
        return []
    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            return parsed
        return [str(parsed)]
    except Exception:
        # fallback for weird comma-separated strings
        return [item.strip() for item in x.strip("[]").split(",") if item.strip()]

if "matched_skills" in training_df.columns:
    training_df["matched_skills"] = training_df["matched_skills"].apply(safe_parse_list)
else:
    training_df["matched_skills"] = [[] for _ in range(len(training_df))]

if "missing_skills" in training_df.columns:
    training_df["missing_skills"] = training_df["missing_skills"].apply(safe_parse_list)
else:
    training_df["missing_skills"] = [[] for _ in range(len(training_df))]

print("Loaded training_df shape:", training_df.shape)

# -------------------------
# 3) HELPERS
# -------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s\+\#\.\-/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def predict_match_score(resume_text, job_text):
    combined = "resume: " + clean_text(resume_text) + " job: " + clean_text(job_text)
    pred = model.predict([combined])[0]
    prob = model.predict_proba([combined])[0][1]
    return int(pred), float(prob)

def build_prompt(resume_text, job_text, score, matched_skills, missing_skills):
    matched_skills_text = ", ".join(matched_skills) if matched_skills else "None identified"
    missing_skills_text = ", ".join(missing_skills[:15]) if missing_skills else "None identified"

    return f"""
You are an expert resume optimizer and career coach.

Analyze how well this resume matches the job.

MATCH SCORE: {round(score * 100, 2)}%

MATCHED SKILLS:
{matched_skills_text}

MISSING SKILLS:
{missing_skills_text}

----------------------
RESUME:
{str(resume_text)[:4000]}

----------------------
JOB DESCRIPTION:
{str(job_text)[:4000]}

----------------------

Provide:

1. Overall fit assessment (2-3 sentences)
2. Top 3 strengths of the candidate
3. Top 3 gaps or weaknesses
4. 5 keywords the candidate should add if truthful
5. 3 rewritten resume bullet points tailored to the job

Be specific, concise, and realistic. Do not invent experience.
"""

def groq_resume_feedback(resume_text, job_text, matched_skills, missing_skills):
    pred, score = predict_match_score(resume_text, job_text)

    prompt = build_prompt(
        resume_text=resume_text,
        job_text=job_text,
        score=score,
        matched_skills=matched_skills,
        missing_skills=missing_skills
    )

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": "You are a resume optimization assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return {
        "prediction": pred,
        "match_score": score,
        "llm_feedback": response.choices[0].message.content
    }

# -------------------------
# 4) RUN ONE TEST EXAMPLE
# -------------------------
sample = training_df.iloc[0]

result = groq_resume_feedback(
    resume_text=sample["resume_text"],
    job_text=sample["job_text"],
    matched_skills=sample["matched_skills"],
    missing_skills=sample["missing_skills"]
)

print("\nPrediction:", result["prediction"])
print("Match score:", round(result["match_score"] * 100, 2), "%")
print("\n===== GROQ FEEDBACK =====\n")
print(result["llm_feedback"])

Loaded training_df shape: (72, 11)

Prediction: 0
Match score: 35.11 %

===== GROQ FEEDBACK =====

**Overall Fit Assessment**
The candidate's resume matches the job description at 35.11%, indicating a moderate level of alignment. While the candidate has some relevant skills, such as leadership and analytical abilities, there are significant gaps in their experience and skills that are crucial for the Summer Analyst Internship role. With some tailoring and emphasis on relevant skills, the candidate may be able to improve their fit for the position.

**Top 3 Strengths of the Candidate**
1. **Leadership experience**: The candidate has demonstrated leadership skills through their experience as a tutor, VP of Membership, and Club President, which can be valuable in a team-based environment.
2. **Analytical skills**: The candidate has a strong foundation in statistical analysis, programming, and data-driven problem-solving, which can be applied to the role's requirements.
3. **Collaborative 

In [2]:
import os
os.environ["GROQ_API_KEY"] = "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"